# DeepWeeds — Độ trễ bổ sung: 7 backbone (batch 1) và các phương pháp suy luận, T4

In [ ]:
import os, sys, platform
REPO_URL = "https://github.com/hdphuc127/K4-Track4-Day2-Deeplearning-Advance.git"
SUB = "submissions/2A202602796-HoDangPhuc/code"
ROOT = "/kaggle/working"
os.chdir(ROOT)
if not os.path.exists("lab"):
    !git clone -q {REPO_URL} lab
else:
    !git -C lab pull -q
CODE = f"{ROOT}/lab/{SUB}"
sys.path.insert(0, CODE)
!pip -q install timm openpyxl
import torch, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import benchmark as BM, model as M, pandas as pd
OUT = f"{ROOT}/out"; os.makedirs(OUT, exist_ok=True)
rows = []
for name in ["resnet50", "resnext50", "convnext_tiny", "deit_small", "swin_tiny", "efficientnet_b0", "mobilenetv3"]:
    m = M.build_model(name, pretrained=False)   # độ trễ không phụ thuộc giá trị trọng số
    for dt in ["fp32", "fp16"]:
        r = BM.latency_report(m, 1, 224, dt)
        rows.append({"nhóm": "backbone", "config": name, **r}); print(name, dt, round(r["p50"], 2), round(r["p95"], 2))
pd.DataFrame(rows).to_csv(f"{OUT}/latency_backbones.csv", index=False)

In [ ]:
m = M.build_model("convnext_tiny", pretrained=False)
rows = []
def add(cfgname, k, size, bs=1):
    r = BM.latency_report(m, bs * k, size, "fp32")      # K view = batch K (cách triển khai thực tế)
    rows.append({"nhóm": "inference", "config": cfgname, "K": k, "img_size": size, **r}); print(cfgname, round(r["p50"], 2), round(r["p95"], 2))
add("I00 1-view 224", 1, 224)
add("I01 flip (K=2) 224", 2, 224)
add("I02b 5crop+flip 192 (K=10)", 10, 192)
add("I04 res256", 1, 256)
add("I04 res288 (F01)", 1, 288)
add("I04 res320", 1, 320)
for sz in (224, 288):
    for dt in ("fp16",):
        r = BM.latency_report(m, 1, sz, dt); rows.append({"nhóm": "inference", "config": f"1-view {sz} {dt}", "K": 1, "img_size": sz, **r})
pd.DataFrame(rows).to_csv(f"{OUT}/latency_inference.csv", index=False)

In [ ]:
!ls /kaggle/working/out